In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week9-lesson-2"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
order_schema= 'order_id long, order_date string , customer_id long, order_status string'

In [3]:
orders_df = spark.read.format('csv').schema(order_schema).load('sparkwriter')

In [4]:
orders_df.createOrReplaceTempView("orders")

In [5]:
spark.sql("select count(*) from orders where order_status = 'CLOSED'")

count(1)
5667000


In [6]:
spark.sql("select count(distinct order_status ) from orders ")

count(DISTINCT order_status)
9


## PartitionBy writer 

In [7]:
orders_df.write \
.mode('overwrite') \
.format('csv') \
.partitionBy("order_status") \
.option('path','/user/itv027484/sparkwriterPart') \
.save()

In [8]:
# [itv027484@g02 ~]$ hadoop fs -ls -h sparkwriterPart
# 3Found 10 items
# -rw-r--r--   3 itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/_SUCCESS
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=CANCELED
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=CLOSED
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=COMPLETE
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=ON_HOLD
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=PAYMENT_REVIEW
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=PENDING
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=PENDING_PAYMENT
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=PROCESSING
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 13:41 sparkwriterPart/order_status=SUSPECTED_FRAUD

### create new DF from partitioned files

In [9]:
orders_df1 = spark.read.format('csv').schema(order_schema).load('sparkwriterPart')

### runs much faster as only 1 partition will be read

In [10]:
orders_df1.filter("order_status = 'CLOSED'").count()  ## runs much faster as only 1 partition will be read

5667000

In [11]:
orders_df1.filter("order_status = 'CLOSED' and customer_id = 8827").count()

750

## partitionBy - more than one col

In [12]:
cust_df = spark.read.format('csv').option('inferSchema','true').load('/public/trendytech/retail_db/customers/part-00000')

In [13]:
cust_df.show(3)

+---+-------+---------+---------+---------+--------------------+-----------+---+-----+
|_c0|    _c1|      _c2|      _c3|      _c4|                 _c5|        _c6|_c7|  _c8|
+---+-------+---------+---------+---------+--------------------+-----------+---+-----+
|  1|Richard|Hernandez|XXXXXXXXX|XXXXXXXXX|  6303 Heather Plaza|Brownsville| TX|78521|
|  2|   Mary|  Barrett|XXXXXXXXX|XXXXXXXXX|9526 Noble Embers...|  Littleton| CO|80126|
|  3|    Ann|    Smith|XXXXXXXXX|XXXXXXXXX|3422 Blue Pioneer...|     Caguas| PR|  725|
+---+-------+---------+---------+---------+--------------------+-----------+---+-----+
only showing top 3 rows



In [14]:
cust_df1 = cust_df.toDF("cust_id","cust_fname","cust_lname","cust_email","cust_pass","cust_addr1","cust_city","cust_state","cust_zip")

In [15]:
cust_df1.show(2)

+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
|cust_id|cust_fname|cust_lname|cust_email|cust_pass|          cust_addr1|  cust_city|cust_state|cust_zip|
+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
|      1|   Richard| Hernandez| XXXXXXXXX|XXXXXXXXX|  6303 Heather Plaza|Brownsville|        TX|   78521|
|      2|      Mary|   Barrett| XXXXXXXXX|XXXXXXXXX|9526 Noble Embers...|  Littleton|        CO|   80126|
+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
only showing top 2 rows



In [16]:
cust_df1.write \
.mode('overwrite') \
.format('csv') \
.partitionBy("cust_state","cust_city") \
.option('path','/user/itv027484/cust_df_2part') \
.save()

In [17]:
# [itv027484@g02 ~]$ hadoop fs -ls cust_df_2part
# Found 45 items
# -rw-r--r--   3 itv027484 supergroup          0 2026-08-04 14:03 cust_df_2part/_SUCCESS
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 14:02 cust_df_2part/cust_state=AL
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 14:02 cust_df_2part/cust_state=AR
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 14:02 cust_df_2part/cust_state=AZ

In [18]:
# [itv027484@g02 ~]$ hadoop fs -ls cust_df_2part/cust_state=WV
# Found 2 items
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 14:03 cust_df_2part/cust_state=WV/cust_city=Martinsburg
# drwxr-xr-x   - itv027484 supergroup          0 2026-08-04 14:03 cust_df_2part/cust_state=WV/cust_city=Wheeling

In [19]:
cust_df2 = spark.read.format('csv').option('inferSchema','true').load('/user/itv027484/cust_df_2part')

## Partition Pruning

### runs fast as the underlying df is partitioned on the same cols as in the filter

In [20]:
cust_df2.filter("cust_state = 'WV' and cust_city = 'Martinsburg'").count()  ## runs fast as the underlying df is partitioned on the same cols as 

9

In [21]:
cust_df2.filter("cust_state = 'WV' ").count()

16

In [23]:
cust_df1.rdd.getNumPartitions()

1

In [24]:
cust_df2.rdd.getNumPartitions()

19